# Elliptic Curves Pseudo Random Number Generators

# Initial Functions

In [62]:
# Finds the m- ary expansion of n
def getExpansion (n ,m):
    listOfDigits =[]
    while n >= m:
        digit =n%m
        listOfDigits . append ( digit )
        n =(n - digit ) // m
    listOfDigits.append(n)
    return listOfDigits
    
def intToText (n):
    t= getExpansion(n ,256)
    myString =''
    for i in t :
        myString = myString + chr(i)
    return myString

def textToInt (s):
    n =0
    k =0
    for i in s :
        n=n +ord( i) *(256** k)
        k=k +1
    return n

# Output the multiplicative inverse of a modulo p
def multInverse (a , p):
    result = extendedGCD (a , p)
    if result [0]!=1: # Error message if a and p are not relatively prime
        s=" Numbers needs to be relatively prime "
        return s
    inv = result [1]% p
    return inv

#extended euclidean algorithm
# Output [r,s,t] satisfying s*a+t*b=r=gcd(a,b)
def extendedGCD(a , b):
    r0 , r=a ,b
    s0 , s =1 ,0
    t0 , t =0 ,1
    while (r >0):
        tempr , temps , tempt =r ,s ,t
        q= r0 // r
        r ,s , t=r0 - q*r ,s0 -q*s ,t0 - q*t
        r0 , s0 , t0 = tempr , temps , tempt
    return [r0 ,s0 , t0 ]

def fast2Power (a ,n ,m):
    res = 1
    while n > 0:
        if n % 2 == 1: #If the bit is 1 multiply by the corresponding square
            res = ( res * a ) % m
        a =( a * a) % m
        n = n // 2
    return res
    
def fast2Power (a ,n ,m):
    res = 1
    while n > 0:
        if n % 2 == 1: #If the bit is 1 multiply by the corresponding square
            res = ( res * a ) % m
        a =( a * a) % m
        n = n // 2
    return res

In [23]:
def findSquareRoot (N ,p) :
    N0 = N % p
    if fast2Power (N0, (p - 1) // 2 ,p) == 1: #Euler ’s criterion
        if p % 4 == 3:
            x1 = fast2Power (N0, (p + 1) // 4 , p) #see assignment 2 exercise 2 theoretical part
            y1 = p - x1
            return [x1 , y1]
        else :
            for i in range(1 , (( p - 1) // 2) + 1):
                if (i * i) % p == N0:
                    x1 = i
                    y1 = p - i
                    return [x1 , y1]
    return []
    
def generateCurve (E , p):
    if isElliptic (E , p) == False :
        print (" This is not an elliptic curve ")
        return None
    A, B = E
    listOfPoints =["O"]
    for x in range (p):
        a =(x**3 + A*x + B) % p
        if a == 0:
            listOfPoints.append ([x ,0])
        if fast2Power (a, (p - 1) // 2, p) == 1: # Euler ’s criterion there are solutions
            y1, y2 = findSquareRoot(a, p)
            listOfPoints.append([x, y1])
            listOfPoints.append([x, y2])
    return listOfPoints

def isElliptic (E, p):
    A, B = E
    discr = (4*( A **3) +27*( B **2) )%p
    return discr != 0

def pointOnCurve (P ,E ,p) :
    if P == "O":
        return True
    else :
        A, B = E
        x, y = P
        return (y **2) %p ==( x **3+ A* x+B) %p

def addPoints(P,Q,E,N):
    A = E[0]
    B = E[1]
    if P == "O":
        return Q
    elif Q == "O":
        return P
    x1, x2 = P[0], Q[0]
    y1, y2 = P[1], Q[1]
    if x1 == x2 % N and y1 == -y2 % N:
        return "O"
    else:
        if P != Q:
            d1 = extendedGCD(x2 - x1, N)[0]
            if d1 != 1:
                return [-1, d1]
            lmbda = (y2 - y1) * multInverse(x2 - x1, N) % N
        else:
            d2 = extendedGCD(2 * y1, N)[0]
            if d2 != 1:
                return [-1, d2]
            lmbda = (3 * fast2Power(x1, 2, N) + A) * multInverse(2 * y1, N) % N
        x3 = (fast2Power(lmbda, 2, N) - x1 - x2) % N
        y3 = (lmbda * (x1 - x3) - y1) % N
        return [x3, y3]
        
def doubleAndAdd(P, n, E, p):
    res = "O"
    while n > 0:
        if n % 2 == 1:
            res = addPoints(res, P, E, p)
        P = addPoints(P, P, E, p)
        n = n // 2
    return res  

In [53]:
E = [5, 12]
p = 13

In [57]:
L = generateCurve(E, p)

In [33]:
len(L)

8

In [35]:
L

['O', [0, 5], [0, 8], [2, 2], [2, 11], [7, 0], [10, 3], [10, 10]]

In [39]:
G = L[1]
G

[0, 5]

# Linear congruential generator

In [68]:
U_0 = [0, 8]
N = 20
U = [addPoints(doubleAndAdd(G, k, E, p), U_0, E, p) for k in range(N)]

In [64]:
doubleAndAdd(G, 2, E, p)

[10, 3]

In [78]:
U[0:8]

[[0, 8], 'O', [0, 5], [10, 3], [2, 11], [7, 0], [2, 2], [10, 10]]

In [80]:
L

['O', [0, 5], [0, 8], [2, 2], [2, 11], [7, 0], [10, 3], [10, 10]]

# Power generator

In [83]:
U_0 = G
e = 7
N = 20
U = [doubleAndAdd(U_0, e**k, E, p) for k in range(N)]

In [87]:
U[0:8]

[[0, 5], [0, 8], [0, 5], [0, 8], [0, 5], [0, 8], [0, 5], [0, 8]]